# Flu Shot Learning — Model Comparison
## Logistic Regression vs Random Forest (H1N1 & Seasonal)

**Author:** Marcelo da Fonseca Oliveira  
**Course:** Applied Machine Learning — City College Dublin, 2026  

This notebook compares baseline Logistic Regression and Random Forest models for both target variables, reporting ROC-AUC, F1, precision and recall.

**Dataset:** [DrivenData — Flu Shot Learning](https://www.drivendata.org/competitions/66/flu-shot-learning/)


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

features = pd.read_csv("training_set_features.csv")
labels = pd.read_csv("training_set_labels.csv")

print("Features columns:", list(features.columns)[:10])
print("Labels columns:", list(labels.columns))

# Detect respondent column
if "respondent_id" in features.columns:
    respondent_col = "respondent_id"
elif "respondentid" in features.columns:
    respondent_col = "respondentid"
else:
    raise ValueError("Could not find respondent_id / respondentid in features file.")

if respondent_col not in labels.columns:
    raise ValueError(f"Column {respondent_col} is not present in labels file.")

# ==========================
# 2. Set index
# ==========================
features.set_index(respondent_col, inplace=True)
labels.set_index(respondent_col, inplace=True)

# ==========================
# 3. Join data
# ==========================
data = features.join(labels, how="inner")
print("Joined data shape:", data.shape)

# Check targets
for col in ["h1n1_vaccine", "seasonal_vaccine"]:
    if col not in data.columns:
        raise ValueError(f"Target column '{col}' not found.")

y_h1n1 = data["h1n1_vaccine"]
y_seasonal = data["seasonal_vaccine"]
X = data.drop(columns=["h1n1_vaccine", "seasonal_vaccine"])

print("X shape before encoding:", X.shape)

# ==========================
# 4. One-hot encoding
# ==========================
X = pd.get_dummies(X, drop_first=True)

# Handle missing values
X = X.fillna(0)

print("X shape after encoding:", X.shape)

# ==========================
# 5. Train/test split
# ==========================
RANDOM_STATE = 42  # for reproducibility

X_train, X_test, y_h1n1_train, y_h1n1_test, y_seasonal_train, y_seasonal_test = train_test_split(
    X,
    y_h1n1,
    y_seasonal,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_h1n1  # remove if it gives error
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ==========================
# 6. Scaling
# ==========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================
# 7. Evaluation function
# ==========================
def evaluate_model(model_name, target_name, y_true, proba):
    preds = (proba >= 0.5).astype(int)
    return {
        "target": target_name,
        "model": model_name,
        "roc_auc": roc_auc_score(y_true, proba),
        "f1": f1_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall": recall_score(y_true, preds),
    }

results = []

# ==========================
# 8. Logistic Regression – h1n1
# ==========================
log_reg_h1n1 = LogisticRegression(max_iter=1000)
log_reg_h1n1.fit(X_train_scaled, y_h1n1_train)

proba_h1n1_lr = log_reg_h1n1.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("LogisticRegression", "h1n1_vaccine", y_h1n1_test, proba_h1n1_lr))

# ==========================
# 9. Random Forest – h1n1
# ==========================
rf_h1n1 = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_h1n1.fit(X_train, y_h1n1_train)

proba_h1n1_rf = rf_h1n1.predict_proba(X_test)[:, 1]
results.append(evaluate_model("RandomForest", "h1n1_vaccine", y_h1n1_test, proba_h1n1_rf))

# ==========================
# 10. Logistic Regression – seasonal
# ==========================
log_reg_seasonal = LogisticRegression(max_iter=1000)
log_reg_seasonal.fit(X_train_scaled, y_seasonal_train)

proba_seasonal_lr = log_reg_seasonal.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("LogisticRegression", "seasonal_vaccine", y_seasonal_test, proba_seasonal_lr))

# ==========================
# 11. Random Forest – seasonal
# ==========================
rf_seasonal = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_seasonal.fit(X_train, y_seasonal_train)

proba_seasonal_rf = rf_seasonal.predict_proba(X_test)[:, 1]
results.append(evaluate_model("RandomForest", "seasonal_vaccine", y_seasonal_test, proba_seasonal_rf))

# ==========================
# 12. Results
# ==========================
results_df = pd.DataFrame(results)
print(results_df)

results_df.to_csv("model_metrics_summary.csv", index=False)

Features columns: ['respondent_id', 'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face']
Labels columns: ['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']
Joined data shape: (26707, 37)
X shape before encoding: (26707, 35)
X shape after encoding: (26707, 93)
Train shape: (21365, 93)
Test shape: (5342, 93)
             target               model   roc_auc        f1  precision  \
0      h1n1_vaccine  LogisticRegression  0.842491  0.570981   0.700384   
1      h1n1_vaccine        RandomForest  0.851099  0.557682   0.734870   
2  seasonal_vaccine  LogisticRegression  0.852616  0.761481   0.776723   
3  seasonal_vaccine        RandomForest  0.854347  0.764257   0.773577   

     recall  
0  0.481938  
1  0.449339  
2  0.746825  
3  0.755159  
